# 06 · TensorRT & Jetson deployment

**Run the code cells on the Jetson.** The text is a field guide you can read anywhere.

## 0. The Jetson mental model

* **Unified memory:** CPU and GPU share the same 8 GB. `torch.cuda.memory_allocated()` does not show what the OS, ROS and camera use. Watch `tegrastats` / `jtop` instead.
* **Power modes:** `nvpmodel` caps watts and clocks. Always report *which mode* a benchmark used. Orin Nano: 7W / 15W / 25W (*MAXN SUPER* on JetPack 6.2+).
* **JetPack pins the stack:** CUDA, cuDNN and TensorRT versions come with the JetPack release. TensorRT engines are **not portable**, so build them *on* the device (or one with an identical GPU + TensorRT version).
* **aarch64:** many pip wheels do not exist for it. Use **jetson-containers** / NGC images or the Jetson AI Lab pip index.

## 1. Setup checklist

```bash
# on the Jetson
cat /etc/nv_tegra_release                # L4T / JetPack version
sudo nvpmodel -q                         # current power mode (see /etc/nvpmodel.conf for ids)
sudo nvpmodel -m <MAXN id> && sudo jetson_clocks   # max perf for benchmarking
sudo systemctl set-default multi-user.target        # optional: no desktop -> ~1 GB more RAM
sudo pip3 install -U jetson-stats && jtop           # monitoring TUI (or: tegrastats)

git clone https://github.com/dusty-nv/jetson-containers && bash jetson-containers/install.sh
git clone https://github.com/<you>/edge-vlm-robot && cd edge-vlm-robot
jetson-containers run -v $PWD:/workspace $(autotag l4t-pytorch)
# inside the container:
cd /workspace && pip install -e . && pip install onnx
```

See also `deploy/jetson/setup_jetson.sh`.

In [ ]:
import sys, pathlib, subprocess, platform
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import torch
print('machine:', platform.machine(), '| torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
IS_JETSON = pathlib.Path('/etc/nv_tegra_release').exists()
if IS_JETSON:
    print(pathlib.Path('/etc/nv_tegra_release').read_text().strip())
    print(subprocess.run(['nvpmodel', '-q'], capture_output=True, text=True).stdout)
else:
    print('Not a Jetson - code cells below will be skipped where hardware is required.')
try:
    import tensorrt as trt
    print('TensorRT', trt.__version__)
except ImportError:
    trt = None
    print('TensorRT python bindings not found')

## 2. Baseline: PyTorch on the Jetson

Always measure the simple path first. It is your reference for correctness *and* speed.

In [ ]:
from edge_vlm.train import load_or_train, VLMPolicy, get_device
from edge_vlm.agent import evaluate_policy
import time

device = get_device()
model, tok = load_or_train(ROOT / 'checkpoints' / 'tiny_robot_vlm.pt', device=device)
model = model.to(device)
if device.type == 'cuda':
    model = model.half()
policy = VLMPolicy(model, tok, device=device)
t = time.time()
print(evaluate_policy(policy, n_episodes=50, seed=777), f'{(time.time() - t) / 50 * 1000:.0f} ms/episode')

## 3. ONNX → TensorRT engines

The ONNX files come from notebook 05 (or `python scripts/export_onnx.py`). Building engines:

```bash
bash deploy/jetson/build_trt_engines.sh exports/ engines/
```

which runs `trtexec` with `--fp16` and **optimization profiles** (min/opt/max shapes) for the dynamic sequence length. TensorRT tunes kernels for the `opt` shape, so set it to your typical prompt length.

Useful `trtexec` flags to explore:
* `--fp16`, `--int8` (needs calibration or Q/DQ nodes), `--best`
* `--memPoolSize=workspace:1024M` (limit build memory on 8 GB devices)
* `--useCudaGraph` (removes launch overhead, a big deal for tiny models)
* `--dumpProfile --separateProfileRun` (per-layer timings)

In [ ]:
ENGINES = ROOT / 'engines'
if IS_JETSON and not (ENGINES / 'decoder_fp16.engine').exists():
    if not (ROOT / 'exports' / 'decoder.onnx').exists():
        import copy
        from edge_vlm.export import export_onnx
        export_onnx(copy.deepcopy(model), ROOT / 'exports')  # export moves to fp32/CPU, so use a copy
    print(subprocess.run(['bash', str(ROOT / 'deploy/jetson/build_trt_engines.sh'), str(ROOT / 'exports'), str(ENGINES)],
                         capture_output=True, text=True).stdout[-2000:])
print(sorted(p.name for p in ENGINES.glob('*.engine')) if ENGINES.exists() else 'no engines yet')

## 4. Run the engines from Python and drive the robot

`deploy/jetson/trt_runner.py` wraps the TensorRT 10 API (`deserialize_cuda_engine`, `set_input_shape`, `set_tensor_address`, `execute_async_v3`) and exposes the same `.act()` interface as the PyTorch policy. The agent cannot tell the difference.

In [ ]:
if trt is not None and (ENGINES / 'decoder_fp16.engine').exists():
    sys.path.insert(0, str(ROOT / 'deploy' / 'jetson'))
    from trt_runner import TRTPolicy
    from edge_vlm.data import generate_samples
    trt_policy = TRTPolicy(ENGINES, tok, num_image_tokens=model.cfg.num_image_tokens)
    s_img = generate_samples(1, seed=1)[0]
    print('PyTorch :', policy.predict(s_img.image, s_img.prompt))
    print('TensorRT:', trt_policy.predict(s_img.image, s_img.prompt))
    t = time.time()
    print(evaluate_policy(trt_policy, n_episodes=50, seed=777), f'{(time.time() - t) / 50 * 1000:.0f} ms/episode')
else:
    print('skipped (needs TensorRT + engines)')

## 5. Deploying a *real* small VLM (SmolVLM from notebook 04)

You have three realistic options on an Orin Nano, from easiest to fastest:

**A. PyTorch + transformers in a jetson-container.** Copy `checkpoints/smolvlm256m_robot/` over and reuse `HFPolicy` from notebook 04. This gives you a correctness baseline.

**B. llama.cpp with GGUF (fast, mature, INT4/INT8).**
```bash
# on the desktop, in a llama.cpp checkout:
python convert_hf_to_gguf.py checkpoints/smolvlm256m_robot --outfile robot-f16.gguf
python convert_hf_to_gguf.py checkpoints/smolvlm256m_robot --mmproj --outfile mmproj-robot-f16.gguf
./llama-quantize robot-f16.gguf robot-q8_0.gguf Q8_0
# on the Jetson (llama.cpp built with CUDA, or the jetson-containers image):
llama-server -m robot-q8_0.gguf --mmproj mmproj-robot-f16.gguf -ngl 99 --port 8080
```
The robot then sends the camera image + prompt to `localhost:8080` (an OpenAI-compatible API). You can try the stock model first with `llama-server -hf ggml-org/SmolVLM-256M-Instruct-GGUF`.

**C. TensorRT Edge-LLM (NVIDIA's production path for Physical AI).** [github.com/NVIDIA/TensorRT-Edge-LLM](https://github.com/NVIDIA/TensorRT-Edge-LLM). Export and quantize the HF checkpoint (INT4 AWQ / FP8 / NVFP4 depending on the GPU) on an x86 machine, build the engines on the Jetson, and run them with the C++ runtime (no Python). Check its supported-model list. Qwen-VL-family VLMs and Cosmos Reason are the typical targets, so a SmolVLM-based model may need option B instead.

## 6. What to measure (make a table for every change)

| Metric | Tool |
|---|---|
| Latency per stage (vision / prefill / decode) | `scripts/benchmark.py`, `trtexec` |
| Peak RAM (whole system!) | `tegrastats`, `jtop` |
| Power (W) and energy per action (J) | `tegrastats` VDD_IN rail × latency |
| GPU contention with cuVSLAM / Nav2 running | run the full stack, then re-measure |
| Thermal throttling over 10+ minutes | `jtop`; check clocks don't drop |
| **Task success** after every optimization | `evaluate_policy` (closed loop!) |

In [ ]:
# Sample tegrastats for 5 seconds while the model runs (Jetson only)
if IS_JETSON:
    import threading
    proc = subprocess.Popen(['tegrastats', '--interval', '500'], stdout=subprocess.PIPE, text=True)
    th = threading.Thread(target=lambda: evaluate_policy(policy, n_episodes=30, seed=1))
    th.start(); th.join(); proc.terminate()
    print(proc.stdout.read()[-1500:])

### Exercises
1. Benchmark at 7W, 15W and 25W. Plot latency vs power and compute energy per action.
2. Rebuild the engines with `--useCudaGraph`. How much does tiny-model latency drop?
3. Run SmolVLM-256M through options A and B. Compare tokens/s against the roofline bound from notebook 00.